In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
import mlflow
import mlflow.sklearn

# 1. Sütun isimleri
index_columns = ['unit_number', 'time_in_cycles']
setting_columns = ['setting_1', 'setting_2', 'setting_3']
sensor_columns = [f'sensor_{i}' for i in range(1, 22)]
columns = index_columns + setting_columns + sensor_columns

# 2. train_FD001.txt verisini okuma
train_df = pd.read_csv(
    '../data/raw/train_FD001.txt', 
    sep=r'\s+', 
    header=None, 
    names=columns
)

# 3. RUL hedef değişkenini hesaplama ve 125 ile kırpma (Piecewise Linear)
max_cycle_df = train_df.groupby('unit_number')['time_in_cycles'].max().reset_index()
max_cycle_df.columns = ['unit_number', 'max_cycle']
train_df = train_df.merge(max_cycle_df, on='unit_number', how='left')

# RUL hesabı
train_df['RUL'] = train_df['max_cycle'] - train_df['time_in_cycles']

# Kırpılmış RUL (Tavan = 125)
MAX_RUL = 125
train_df['RUL_clipped'] = train_df['RUL'].clip(upper=MAX_RUL)

print("Veri başarıyla yüklendi ve RUL sütunları eklendi.")
print(f"Tablo Boyutu: {train_df.shape}")
train_df[['unit_number', 'time_in_cycles', 'RUL', 'RUL_clipped']].head()

Veri başarıyla yüklendi ve RUL sütunları eklendi.
Tablo Boyutu: (20631, 29)


,unit_number,time_in_cycles,RUL,RUL_clipped
0,1,1,191,125
1,1,2,190,125
2,1,3,189,125
3,1,4,188,125
4,1,5,187,125


In [2]:
#veri sızıntısı(data leakage'yi önlemek için satır bazlı değil motor bazlı bölme yapmak uygun olur.)


In [3]:
# 1. Benzersiz motor kimliklerini alalım
unique_units = train_df['unit_number'].unique()

In [4]:
# 2. %80 Train, %20 Validation olarak ayıralım (random_state=42)
train_units, val_units = train_test_split(unique_units, test_size=0.20, random_state=42)

In [5]:
# 3. Veriyi bu motor listelerine göre ikiye bölelim
train_data = train_df[train_df['unit_number'].isin(train_units)].copy()
val_data = train_df[train_df['unit_number'].isin(val_units)].copy()

In [7]:
# 4. Kontroller
print(f"Train Motor Sayısı: {len(train_units)} | Toplam Satır: {len(train_data)}")
print(f"Val Motor Sayısı:   {len(val_units)} | Toplam Satır: {len(val_data)}")

Train Motor Sayısı: 80 | Toplam Satır: 16561
Val Motor Sayısı:   20 | Toplam Satır: 4070


In [8]:
# Sızıntı kontrolü (ortak motor olmamalı -> 0 çıkmalı)
intersection = set(train_data['unit_number']).intersection(set(val_data['unit_number']))
print(f"Ortak Motor Sayısı (Sızıntı Kontrolü): {len(intersection)}")

Ortak Motor Sayısı (Sızıntı Kontrolü): 0


In [9]:
# 1. Elenecek gereksiz sensörler ve ayarlar
constant_features = [
    'sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 
    'sensor_16', 'sensor_18', 'sensor_19', 'setting_3'
]

In [10]:
# 2. Modele özellik (X) olarak girmeyecek sütunlar (ID ve hedef sütunları)
metadata_and_target = ['unit_number', 'max_cycle', 'RUL', 'RUL_clipped']

In [11]:
# 3. Modele girecek temiz özellik listesini belirleme
feature_cols = [
    col for col in train_data.columns 
    if col not in constant_features and col not in metadata_and_target
]

In [12]:
print(f"Kullanılacak Özellik Sayısı: {len(feature_cols)}")
print("Seçilen Özellikler:", feature_cols)

Kullanılacak Özellik Sayısı: 17
Seçilen Özellikler: ['time_in_cycles', 'setting_1', 'setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


In [13]:
# 4. X ve y matrislerini oluşturma
X_train = train_data[feature_cols]
y_train = train_data['RUL_clipped']

X_val = val_data[feature_cols]
y_val = val_data['RUL_clipped']

print(f"\nX_train Boyutu: {X_train.shape} | y_train Boyutu: {y_train.shape}")
print(f"X_val Boyutu:   {X_val.shape} | y_val Boyutu:   {y_val.shape}")


X_train Boyutu: (16561, 17) | y_train Boyutu: (16561,)
X_val Boyutu:   (4070, 17) | y_val Boyutu:   (4070,)


#### Baseline Model

In [16]:
import os
# MLflow'un yerel dosya sistemine (mlruns/) yazmasına izin veriyoruz
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
import mlflow
import mlflow.sklearn

In [17]:
# 1. MLflow takip dizini
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("turbofan-rul-baseline")

2026/09/02 09:38:56 INFO mlflow.tracking.fluent: Experiment with name 'turbofan-rul-baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///C:/Users/dell/Desktop/turbofan-rul/notebooks/../mlruns/863826732218489687', creation_time=1788331136580, effective_trace_archival_retention=None, experiment_id='863826732218489687', last_update_time=1788331136580, lifecycle_stage='active', name='turbofan-rul-baseline', tags={}, trace_location=None, workspace='default'>

In [18]:
# 2. Pipeline tanımı (Ölçekleme + Doğrusal Regresyon)
baseline_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

In [20]:
# 3. MLflow ile deneyi başlatma
def compute_phm_score(y_true, y_pred):
    """
    NASA PHM 2008 Challenge asimetrik ceza skoru (Vektörize).
    d = y_pred - y_true
    d < 0  -> Erken tahmin: exp(-d / 13) - 1
    d >= 0 -> Geç tahmin:   exp(d / 10) - 1
    """
    d = np.array(y_pred) - np.array(y_true)
    scores = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
    return float(np.sum(scores))

# MLflow ile baseline koşusunu metrikle birlikte kaydedelim
with mlflow.start_run(run_name="linear_regression_baseline_with_phm"):
    baseline_pipeline.fit(X_train, y_train)
    
    y_train_pred = baseline_pipeline.predict(X_train)
    y_val_pred = baseline_pipeline.predict(X_val)
    
    # Standart metrikler
    train_rmse = root_mean_squared_error(y_train, y_train_pred)
    val_rmse = root_mean_squared_error(y_val, y_val_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    
    # NASA Asimetrik PHM skorları
    train_phm = compute_phm_score(y_train, y_train_pred)
    val_phm = compute_phm_score(y_val, y_val_pred)
    
    # Parametreler
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("scaler", "StandardScaler")
    mlflow.log_param("feature_count", len(feature_cols))
    mlflow.log_param("max_rul_clip", MAX_RUL)
    
    # Metrikler
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_r2", val_r2)
    mlflow.log_metric("train_phm_score", train_phm)
    mlflow.log_metric("val_phm_score", val_phm)
    
    mlflow.sklearn.log_model(baseline_pipeline, name="model")
    
    print(f"Val RMSE:      {val_rmse:.2f}")
    print(f"Val MAE:       {val_mae:.2f}")
    print(f"Val R²:        {val_r2:.4f}")
    print(f"Val PHM Score: {val_phm:,.2f}")

2026/09/02 11:49:38 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\dell\AppData\Local\Temp\tmp5c7e2rxd\model\model.skops, flavor: sklearn). Fall back to return ['scikit-learn==1.9.0', 'skops==0.14.0']. Set logging level to DEBUG to see the full traceback. 


Val RMSE:      17.93
Val MAE:       14.53
Val R²:        0.8152
Val PHM Score: 21,964.17
